In [2]:
import numpy as np
import cv2
import matplotlib.pyplot as plt


In [22]:
import cv2
import numpy as np
import itertools as it
import matplotlib.pyplot as plt
import matplotlib as mpl
from cv02504.util_functions import *
from cv02504.exam_toolkit.epipolar import *

In [3]:
def ensure_column(x, name: str = "x") -> np.ndarray:
    """Return x as a float column vector of shape (n, 1)."""
    arr = np.asarray(x, dtype=float)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    elif arr.ndim == 2 and arr.shape[0] == 1:
        arr = arr.T
    if arr.ndim != 2 or arr.shape[1] != 1:
        raise ValueError(f"{name} must be a vector convertible to shape (n,1). Got {arr.shape}.")
    return arr

In [7]:
def point_line_distance(line: np.ndarray, point_h: np.ndarray) -> float:
    """Shortest distance from homogeneous point to homogeneous line."""
    l = ensure_column(line, name="line")
    p = ensure_column(point_h, name="point_h")
    if l.shape != (3, 1):
        raise ValueError(f"line must be 3x1. Got {l.shape}.")
    if p.shape != (3, 1):
        raise ValueError(f"point_h must be 3x1. Got {p.shape}.")

    num = float(np.abs((l.T @ p).item()))
    den = float(np.abs(p[2, 0]) * np.sqrt(l[0, 0] ** 2 + l[1, 0] ** 2))
    if den == 0.0:
        raise ValueError("Degenerate line/point configuration gives zero denominator.")
    return num / den


q = t = np.array([[2, 4, 3]]).T
l = t = np.array([[1, 2, 2]]).T

print( point_line_distance(l,q))

2.3851391759997758


In [13]:
k = camera_intrinsic(1720,(680,610))
r = R = cv2.Rodrigues(np.array([-0.1, 0.1, -0.2]))[0]
t = np.array([[0.09], [0.05], [0.05]])
Q = np.array([[-0.03, 0.01, 0.59]]).T

print(project_points(k,r,t,Q))

[[1023.50377104]
 [ 930.29756751]]


In [14]:
K = camera_intrinsic(1400, (750, 520))
R = cv2.Rodrigues(np.array([0.2, 0.2, -0.1]))[0]
t = np.array([[-0.08, 0.01, 0.03]]).T
Q = np.array([[-0.38, 0.1, 1.32]]).T

ans = project_points(K, R, t, Q)
ans

array([[557.52555675],
       [383.75661509]])

In [27]:
K = np.array([[900, 0, 1070], [0, 900, 610.0], [0, 0, 1]], float)
R1 = cv2.Rodrigues(np.array([-1.6, 0.3, -2.1]))[0]
t1 = np.array([[0.0], [1.0], [3.0]], float)
R2 = cv2.Rodrigues(np.array([-0.4, -1.3, -1.6]))[0]
t2 = np.array([[0.0], [1.0], [6.0]], float)
R3 = cv2.Rodrigues(np.array([2.5, 1.7, -0.4]))[0]
t3 = np.array([[2.0], [-7.0], [25.0]], float)

p1 = np.array([[1046.0], [453.0]])  # 2x1
p2 = np.array([[1126.0], [671.0]])
p3 = np.array([[1165.0], [453.0]])

fundmatrix = (fundamental_matrix_from_rt(K,R1,t1,K,R2,t2))

print(point_to_epipolar_line_distance(fundmatrix,p1,p2,2))


(13.271829068148199, array([[ 0.00239413],
       [-0.00659817],
       [ 1.6384222 ]]))


In [30]:
# Converts homogeneous coordinates to inhomogeneous coordinates
def PiInv(coords):
    return coords[:-1]/coords[-1]

In [31]:
def undistort_image_point(p, K, distCoeffs):
    q = PiInv(np.linalg.inv(K)@Pi(p))
    p_cam_distorted = q * (1+(distCoeffs[0]*(np.linalg.norm(q, axis=0))**2)+(distCoeffs[1]*(np.linalg.norm(q, axis=0))**4)+(distCoeffs[2]*(np.linalg.norm(q, axis=0))**6))
    return PiInv(K@Pi(p_cam_distorted))

In [32]:
K = np.array([[300, 0, 840], [0, 300, 620], [0, 0, 1]], float)
k3 = -0.2
k5 = 0.01
k7 = -0.03
p = np.array([[400], [500]])

p_h = undistort_image_point(p, K, [k3, k5, k7])

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 1 is different from 3)

In [41]:
import numpy as np
from matplotlib import pyplot as plt
import cv2
from util_functions_b import *

# Question 4, Answer G
K = np.array([[300, 0, 840], [0, 300, 620], [0, 0, 1]], float)
k3 = -0.2
k5 = 0.01
k7 = -0.03
p = np.array([[400], [500]])

p_h = undistort_image_point(p, K, [k3, k5, k7])
print(p_h)

[[742.81960823]
 [593.49625679]]


In [44]:
# Question 6, Answer G
harris_data = np.load("harris.npy", allow_pickle=True).item()
g_x = harris_data["g*(I_x^2)"]
g_y = harris_data["g*(I_y^2)"]
g_x_y = harris_data["g*(I_x I_y)"]

C = C_x_y = np.stack([
    np.stack([g_x,   g_x_y], axis=-1),
    np.stack([g_x_y, g_y],   axis=-1)
], axis=-2)

corners = cornerDetector("", "", "", 0.06, 5, useC=True, in_C=C,use_tau_as_threshold=True)
print(corners)

[(2, 1)]


In [46]:
sift_data = np.load("sift_data.npy", allow_pickle=True).item()
kp1 = sift_data["kp1"]
des1 = sift_data["des1"]
kp2 = sift_data["kp2"]
des2 = sift_data["des2"]
r = 0.8

bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)

matches = bf.knnMatch(rootsift(des1), rootsift(des2), k=2)
good_matches = []
for m, n in matches:
    if m.distance < r * n.distance:
        good_matches.append(m)

print(len(good_matches))

331
